# Session-Specific Meta-Confidence Models

Instead of one global meta model, we train **3 separate meta classifiers** — one per trading session:
- **London** (`is_london == 1`): 07:00–16:00 UTC
- **NY** (`is_ny == 1`): 13:00–21:00 UTC (includes overlap)
- **Asia** (`is_asia == 1`): 21:00–07:00 UTC

**Rationale:** The global meta model's decision boundary is dominated by Asian/NY-transition patterns.
A London-specific meta model can learn what 'high confidence' looks like *during London hours* without
being overwhelmed by the majority class from other sessions.

**Base quantile models:** Unchanged from `models_6/3_quants/` (Q25, Q50, Q75).
Only the confidence filter changes.

**Models saved to:** `backend/models_7/meta/`

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib
import warnings
import gc
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import roc_auc_score

FEATURES_DIR  = Path('../backend/data/features_6')
QUANT_DIR     = Path('../backend/models_6/3_quants')   # base quantile models — unchanged
META_DIR      = Path('../backend/models_7/meta')
META_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END         = '2024-06-30'
AVG_SPREAD        = 0.00028
MIN_Q50_THRESHOLD = AVG_SPREAD * 0.5
QUANTILES         = [0.25, 0.50, 0.75]
QUANTILE_NAMES    = ['Q25', 'Q50', 'Q75']

print('Setup complete.')
print(f'Quantile models from : {QUANT_DIR}')
print(f'Session meta saved to: {META_DIR}')

## 1. Load Data

In [ ]:
# Load features_6 (all 15 pairs)
dfs = []
for f in sorted(FEATURES_DIR.glob('*_features.parquet')):
    tmp = pd.read_parquet(f)
    dfs.append(tmp)

df_features = pd.concat(dfs).sort_index()
del dfs
print(f'features_6: {df_features.shape}')

# Load labels
df_labels_maj = pd.read_parquet(Path('../backend/data/features_2') / 'all_pairs_microstructure.parquet')[['pair', 'label_1H', 'label_4H']]
df_labels_cro = pd.read_parquet(Path('../backend/data/features_3') / 'all_pairs_microstructure.parquet')[['pair', 'label_1H', 'label_4H']]
df_labels = pd.concat([df_labels_maj, df_labels_cro]).sort_index()
del df_labels_maj, df_labels_cro

# Merge on timestamp + pair
df_features = df_features.reset_index()
df_labels   = df_labels.reset_index()
df = pd.merge(df_features, df_labels, on=[df_features.columns[0], 'pair'], how='inner')
df = df.set_index(df.columns[0]).sort_index()
del df_features, df_labels
print(f'merged:     {df.shape}')

label_cols   = [c for c in df.columns if c.startswith('label_')]
drop_cols    = label_cols + ['pair', 'mfe_long_pips', 'mfe_short_pips', 'trail_long_bars',
                             'trail_short_bars', 'trail_stop_pips', 'mfe_atr_24']
feature_cols = [c for c in df.columns if c not in drop_cols]
TARGET_COL   = 'label_1H'

df_train = df[df.index <= TRAIN_END].copy()
df_test  = df[df.index > TRAIN_END].copy()

y_all      = df_train[TARGET_COL]
valid_mask = y_all.notna()
X_clean    = df_train[feature_cols][valid_mask].ffill().fillna(0)
y_clean    = y_all[valid_mask]
pairs_clean = df_train.loc[valid_mask, 'pair']

print(f'\nTrain: {len(X_clean):,} rows | Test: {len(df_test):,} rows')
print(f'Features: {len(feature_cols)}')
print(f'Session columns present: is_london={"is_london" in feature_cols}, is_ny={"is_ny" in feature_cols}, is_asia={"is_asia" in feature_cols}')

## 2. Regenerate OOF Predictions from models_6 Quantile Models

Same walk-forward CV as notebook 04 in notebooks_6. OOF predictions = no leakage.

In [ ]:
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    test_size = int(n * test_ratio)
    splits = []
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n:
            break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

def get_lgbm_params(quantile):
    return {
        'objective':         'quantile',
        'alpha':             quantile,
        'metric':            'quantile',
        'boosting_type':     'gbdt',
        'n_estimators':      5000,
        'learning_rate':     0.02,
        'num_leaves':        64,
        'max_depth':         6,
        'min_child_samples': 50,
        'feature_fraction':  0.7,
        'bagging_fraction':  0.8,
        'bagging_freq':      5,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'random_state':      42,
        'n_jobs':            -1,
        'verbose':           -1,
        'device':            'gpu',
    }

splits   = walk_forward_splits(len(X_clean))
oof_preds = {q: np.full(len(X_clean), np.nan) for q in QUANTILES}

for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
    print(f'Generating OOF for {q_name}...')
    params = get_lgbm_params(q)
    for fold, (train_idx, test_idx) in enumerate(splits):
        X_tr, y_tr = X_clean.iloc[train_idx], y_clean.iloc[train_idx]
        X_te, y_te = X_clean.iloc[test_idx],  y_clean.iloc[test_idx]
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof_preds[q][test_idx] = model.predict(X_te)
    del model; gc.collect()
    valid_oof = ~np.isnan(oof_preds[q])
    print(f'  {q_name}: {valid_oof.sum():,} OOF predictions')

has_oof = ~np.isnan(oof_preds[0.50])
print(f'\nTotal OOF rows: {has_oof.sum():,} / {len(X_clean):,}')

## 3. Build Meta-Dataset with Session Labels

In [ ]:
# Build OOF meta-dataset
oof_df = X_clean[has_oof].copy()
oof_df['Q25_oof']       = oof_preds[0.25][has_oof]
oof_df['Q50_oof']       = oof_preds[0.50][has_oof]
oof_df['Q75_oof']       = oof_preds[0.75][has_oof]
oof_df['abs_Q50']       = np.abs(oof_df['Q50_oof'])
oof_df['iqr']           = oof_df['Q75_oof'] - oof_df['Q25_oof']
oof_df['conf_ratio']    = oof_df['abs_Q50'] / oof_df['iqr'].clip(lower=1e-10)
oof_df['q50_dir']       = np.sign(oof_df['Q50_oof'])
oof_df['actual_return'] = y_clean.values[has_oof]
oof_df['actual_dir']    = np.sign(oof_df['actual_return'])
oof_df['pair']          = pairs_clean.values[has_oof]
oof_df['q50_correct']   = (oof_df['q50_dir'] == oof_df['actual_dir']).astype(int)

# Filter tradeable zone
tradeable = oof_df[oof_df['abs_Q50'] > MIN_Q50_THRESHOLD].copy()

# Session assignment — a row can belong to multiple sessions (overlap), assign by priority
# We use the features already in the data: is_london, is_ny, is_asia
# For overlap hours, assign to both (each session model trains on its own rows)
print(f'Tradeable OOF rows: {len(tradeable):,}')
print(f'Q50 accuracy overall: {tradeable["q50_correct"].mean():.1%}')
print()

for sess in ['is_london', 'is_ny', 'is_asia']:
    sess_rows = tradeable[tradeable[sess] == 1]
    print(f'{sess}: {len(sess_rows):,} rows | Q50 acc: {sess_rows["q50_correct"].mean():.1%} | '
          f'class balance: {sess_rows["q50_correct"].mean():.1%} correct')

meta_feature_cols = feature_cols + ['Q50_oof', 'Q25_oof', 'Q75_oof', 'abs_Q50', 'iqr', 'conf_ratio']
print(f'\nMeta-features: {len(meta_feature_cols)}')

## 4. Train Session-Specific Meta Models

One LightGBM binary classifier per session. Same params as `models_6` meta, but each trained only on its session's rows.

In [ ]:
meta_params = {
    'objective':         'binary',
    'metric':            'auc',
    'boosting_type':     'gbdt',
    'n_estimators':      3000,
    'learning_rate':     0.01,
    'num_leaves':        32,
    'max_depth':         4,
    'min_child_samples': 30,
    'feature_fraction':  0.6,
    'bagging_fraction':  0.7,
    'bagging_freq':      5,
    'reg_alpha':         0.5,
    'reg_lambda':        0.5,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
    'device':            'gpu',
    'is_unbalance':      True,
}

SESSIONS = {
    'london': 'is_london',
    'ny':     'is_ny',
    'asia':   'is_asia',
}

session_results = {}   # store CV results per session
session_models  = {}   # store trained models

for sess_name, sess_col in SESSIONS.items():
    print(f'\n{"="*55}')
    print(f'SESSION: {sess_name.upper()} ({sess_col})')
    print(f'{"="*55}')

    sess_data = tradeable[tradeable[sess_col] == 1].copy()
    X_meta    = sess_data[meta_feature_cols]
    y_meta    = sess_data['q50_correct']

    print(f'Rows: {len(X_meta):,} | Correct: {y_meta.mean():.1%}')

    if len(X_meta) < 200:
        print(f'  WARNING: Too few rows ({len(X_meta)}) for reliable training. Skipping.')
        continue

    meta_splits      = walk_forward_splits(len(X_meta), n_splits=5, test_ratio=0.1)
    meta_oof_proba   = np.full(len(X_meta), np.nan)
    meta_best_iters  = []
    meta_aucs        = []

    for fold, (train_idx, test_idx) in enumerate(meta_splits):
        X_tr, y_tr = X_meta.iloc[train_idx], y_meta.iloc[train_idx]
        X_te, y_te = X_meta.iloc[test_idx],  y_meta.iloc[test_idx]

        model = lgb.LGBMClassifier(**meta_params)
        model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

        proba = model.predict_proba(X_te)[:, 1]
        meta_oof_proba[test_idx] = proba
        auc = roc_auc_score(y_te, proba)
        meta_aucs.append(auc)
        meta_best_iters.append(model.best_iteration_)
        print(f'  Fold {fold+1}: AUC={auc:.4f}, iters={model.best_iteration_}')

    del model; gc.collect()

    mean_auc  = np.mean(meta_aucs)
    avg_iter  = max(50, int(np.mean(meta_best_iters)))
    print(f'  Mean CV AUC: {mean_auc:.4f} | Avg iters: {avg_iter}')

    # Train final model on all session rows
    final_meta = lgb.LGBMClassifier(**{**meta_params, 'n_estimators': avg_iter})
    final_meta.fit(X_meta, y_meta)

    # Save
    save_path = META_DIR / f'meta_{sess_name}.joblib'
    joblib.dump({
        'model':              final_meta,
        'meta_feature_cols':  meta_feature_cols,
        'min_q50_threshold':  MIN_Q50_THRESHOLD,
        'session':            sess_name,
        'session_col':        sess_col,
        'train_end':          TRAIN_END,
        'cv_auc':             mean_auc,
        'n_iters':            avg_iter,
        'n_train_rows':       len(X_meta),
    }, save_path)

    size_mb = save_path.stat().st_size / 1024 / 1024
    print(f'  Saved: {save_path.name} ({size_mb:.1f} MB)')

    session_results[sess_name] = {
        'cv_auc':          mean_auc,
        'avg_iter':        avg_iter,
        'n_rows':          len(X_meta),
        'oof_proba':       meta_oof_proba,
        'sess_data':       sess_data,
    }
    session_models[sess_name] = final_meta

    del final_meta; gc.collect()

print(f'\nAll session models saved to {META_DIR}')

## 5. OOF Threshold Sweep — Per Session

In [ ]:
for sess_name, res in session_results.items():
    oof_proba  = res['oof_proba']
    sess_data  = res['sess_data']
    has_pred   = ~np.isnan(oof_proba)
    eval_data  = sess_data[has_pred].copy()
    eval_data['meta_proba'] = oof_proba[has_pred]

    print(f'\n--- {sess_name.upper()} (OOF, {len(eval_data):,} rows) ---')
    print(f'{"Threshold":<12} {"Trades":>8} {"WR":>8} {"EV/trade":>12} {"TotalPnL":>10}')
    print('-' * 55)

    for thresh in np.arange(0.50, 0.96, 0.05):
        mask = eval_data['meta_proba'] > thresh
        n = mask.sum()
        if n < 10:
            continue
        s   = eval_data[mask]
        pnl = s['q50_dir'] * s['actual_return'] - AVG_SPREAD
        wr  = s['q50_correct'].mean()
        flag = ' <<<' if wr >= 0.80 and n >= 50 else ''
        print(f'P > {thresh:.2f}    {n:>8,} {wr:>7.1%} {pnl.mean():>12.6f} {pnl.sum():>10.4f}{flag}')

## 6. Test Set Evaluation — Session-Routed Meta Models

Each test row is routed to its session's meta model. For overlap hours (London+NY), both models are checked — we take the **stricter** (lower) probability to be conservative.

In [ ]:
# Prepare test set predictions from models_6 quantile models
X_test       = df_test[feature_cols].ffill().fillna(0)
y_test       = df_test[TARGET_COL]
valid_test   = y_test.notna()
X_test_clean = X_test[valid_test]
y_test_clean = y_test[valid_test]

q_preds_test = {}
for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
    q_int = int(q * 100)
    bundle = joblib.load(QUANT_DIR / f'model_1H_Q{q_int}.joblib')
    q_preds_test[q_name] = bundle['model'].predict(X_test_clean)
    print(f'{q_name}: mean={q_preds_test[q_name].mean():.6f}')

# Build test results df
test_results = X_test_clean.copy()
test_results['Q25_oof']       = q_preds_test['Q25']
test_results['Q50_oof']       = q_preds_test['Q50']
test_results['Q75_oof']       = q_preds_test['Q75']
test_results['abs_Q50']       = np.abs(test_results['Q50_oof'])
test_results['iqr']           = test_results['Q75_oof'] - test_results['Q25_oof']
test_results['conf_ratio']    = test_results['abs_Q50'] / test_results['iqr'].clip(lower=1e-10)
test_results['pred_dir']      = np.sign(test_results['Q50_oof'])
test_results['actual_return'] = y_test_clean.values
test_results['actual_dir']    = np.sign(test_results['actual_return'])
test_results['pair']          = df_test.loc[valid_test, 'pair'].values

# Filter tradeable zone
test_tradeable = test_results[test_results['abs_Q50'] > MIN_Q50_THRESHOLD].copy()
print(f'\nTest tradeable rows: {len(test_tradeable):,}')

In [ ]:
# Apply each session model to its rows on the test set
# Load session models from disk
loaded_models = {}
for sess_name in SESSIONS:
    p = META_DIR / f'meta_{sess_name}.joblib'
    if p.exists():
        loaded_models[sess_name] = joblib.load(p)
        print(f'Loaded {p.name}')

# Route each test row to its session model
# Priority: london -> ny -> asia (for overlap, first match wins)
# We assign a meta_proba to each row, -1 = no model assigned
test_tradeable = test_tradeable.copy()
test_tradeable['meta_proba']   = np.nan
test_tradeable['meta_session'] = ''

for sess_name, sess_col in SESSIONS.items():
    if sess_name not in loaded_models:
        continue
    bundle = loaded_models[sess_name]
    model  = bundle['model']
    feats  = bundle['meta_feature_cols']

    # Rows for this session that don't yet have a meta_proba assigned
    sess_mask = (test_tradeable[sess_col] == 1) & (test_tradeable['meta_proba'].isna())
    sess_rows = test_tradeable[sess_mask]

    if len(sess_rows) == 0:
        continue

    proba = model.predict_proba(sess_rows[feats])[:, 1]
    test_tradeable.loc[sess_mask, 'meta_proba']   = proba
    test_tradeable.loc[sess_mask, 'meta_session'] = sess_name
    print(f'{sess_name}: {len(sess_rows):,} rows assigned | proba mean={proba.mean():.3f}')

# Rows with no session model (shouldn't happen often)
unassigned = test_tradeable['meta_proba'].isna().sum()
print(f'\nUnassigned rows (no session): {unassigned}')
print(f'Total assigned: {(~test_tradeable["meta_proba"].isna()).sum():,}')

In [ ]:
# Evaluate session-routed meta at different thresholds
assigned = test_tradeable[test_tradeable['meta_proba'].notna()].copy()
n_test_days = (assigned.index.max() - assigned.index.min()).days

print(f'SESSION-ROUTED META — TEST SET RESULTS (post {TRAIN_END})')
print(f'\n{"Threshold":<12} {"Trades":>8} {"Tr/day":>8} {"WR":>8} {"EV/trade":>12} {"TotalPnL":>10} {"Sharpe":>8}')
print('-' * 75)

test_sweep = []
for thresh in np.arange(0.50, 0.96, 0.05):
    mask = assigned['meta_proba'] > thresh
    n = mask.sum()
    if n < 10:
        continue
    s      = assigned[mask]
    pnl    = s['pred_dir'] * s['actual_return'] - AVG_SPREAD
    wr     = (s['pred_dir'] == s['actual_dir']).mean()
    ev     = pnl.mean()
    total  = pnl.sum()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    daily  = n / n_test_days
    test_sweep.append({'thresh': thresh, 'trades': n, 'daily': daily, 'wr': wr, 'ev': ev, 'pnl': total, 'sharpe': sharpe})
    flag = ' <<<' if wr >= 0.80 and n >= 100 else ''
    print(f'P > {thresh:.2f}    {n:>8,} {daily:>8.2f} {wr:>7.1%} {ev:>12.6f} {total:>10.4f} {sharpe:>8.2f}{flag}')

# Baseline: global meta from models_6
print(f'\n--- Baseline: global meta (models_6) ---')
global_meta = joblib.load(Path('../backend/models_6/meta/meta_confidence.joblib'))
global_proba = global_meta['model'].predict_proba(test_tradeable[global_meta['meta_feature_cols']])[:, 1]
test_tradeable['global_meta_proba'] = global_proba

for thresh in [0.50, 0.55, 0.60]:
    mask = test_tradeable['global_meta_proba'] > thresh
    n = mask.sum()
    if n < 10: continue
    s = test_tradeable[mask]
    pnl = s['pred_dir'] * s['actual_return'] - AVG_SPREAD
    wr = (s['pred_dir'] == s['actual_dir']).mean()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    print(f'Global P>{thresh:.2f}  {n:>8,} {n/n_test_days:>8.2f} {wr:>7.1%} {pnl.mean():>12.6f} {pnl.sum():>10.4f} {sharpe:>8.2f}')

## 7. Per-Session Breakdown — Where is the Edge Coming From?

In [ ]:
# Pick best threshold from sweep
best = max([r for r in test_sweep if r['wr'] >= 0.70], key=lambda x: x['pnl'], default=None)
if best is None:
    best = max(test_sweep, key=lambda x: x['pnl'])
BEST_THRESH = best['thresh']
print(f'Best threshold: P > {BEST_THRESH:.2f} | Trades: {best["trades"]} | WR: {best["wr"]:.1%} | PnL: {best["pnl"]:.4f}\n')

filtered = assigned[assigned['meta_proba'] > BEST_THRESH]

print(f'BREAKDOWN BY SESSION (P > {BEST_THRESH:.2f})')
print(f'{"Session":<10} {"Trades":>8} {"WR":>8} {"EV/trade":>12} {"TotalPnL":>10} {"Sharpe":>8}')
print('-' * 65)

for sess_name in ['london', 'ny', 'asia']:
    s = filtered[filtered['meta_session'] == sess_name]
    if len(s) < 5:
        print(f'{sess_name:<10} {len(s):>8,}  (too few)')
        continue
    pnl    = s['pred_dir'] * s['actual_return'] - AVG_SPREAD
    wr     = (s['pred_dir'] == s['actual_dir']).mean()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    flag   = ' <<<' if pnl.mean() > 0 else ''
    print(f'{sess_name:<10} {len(s):>8,} {wr:>7.1%} {pnl.mean():>12.6f} {pnl.sum():>10.4f} {sharpe:>8.2f}{flag}')

# Per-hour heatmap
print(f'\nPER-HOUR PnL (P > {BEST_THRESH:.2f}):')
filtered2 = filtered.copy()
filtered2['hour'] = filtered2.index.hour
filtered2['pnl']  = filtered2['pred_dir'] * filtered2['actual_return'] - AVG_SPREAD
hour_pnl = filtered2.groupby('hour')['pnl'].agg(['sum', 'count', 'mean'])
hour_pnl.columns = ['total_pnl', 'trades', 'ev']
print(hour_pnl.to_string())

## 8. Equity Curves — Session-Routed vs Global Meta

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.patch.set_facecolor('#080c14')

plot_configs = [
    ('Session-Routed P>0.50', assigned[assigned['meta_proba'] > 0.50]),
    (f'Session-Routed P>{BEST_THRESH:.2f} (Best)', assigned[assigned['meta_proba'] > BEST_THRESH]),
    ('Global Meta P>0.50 (models_6)', test_tradeable[test_tradeable['global_meta_proba'] > 0.50]),
    ('Global Meta P>0.55 (models_6)', test_tradeable[test_tradeable['global_meta_proba'] > 0.55]),
]

for ax, (title, subset) in zip(axes.flatten(), plot_configs):
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')

    if len(subset) == 0:
        ax.set_title('No data', color='white')
        continue

    pnl     = subset['pred_dir'] * subset['actual_return'] - AVG_SPREAD
    cum_pnl = pnl.cumsum()

    for pair in sorted(subset['pair'].unique()):
        pmask    = subset['pair'] == pair
        pair_pnl = (subset.loc[pmask, 'pred_dir'] * subset.loc[pmask, 'actual_return'] - AVG_SPREAD).cumsum()
        ax.plot(pair_pnl.index, pair_pnl.values, alpha=0.3, linewidth=0.7)

    ax.plot(cum_pnl.index, cum_pnl.values, color='#4fc3f7', linewidth=2)
    ax.axhline(0, color=(1, 1, 1, 0.2), linewidth=1, linestyle='--')

    wr     = (subset['pred_dir'] == subset['actual_dir']).mean()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    ax.set_title(f'{title}\n{len(subset):,} trades | WR: {wr:.1%} | Sharpe: {sharpe:.2f}',
                 color='white', fontsize=9)
    ax.set_ylabel('Cum PnL', color='white', fontsize=8)

plt.suptitle('Session-Routed Meta vs Global Meta — Test Set Equity Curves', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Feature Importance — Per Session Model

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 10))
fig.patch.set_facecolor('#080c14')

quantile_feats = {'Q50_oof', 'Q25_oof', 'Q75_oof', 'abs_Q50', 'iqr', 'conf_ratio'}

for ax, sess_name in zip(axes, ['london', 'ny', 'asia']):
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white', labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')

    p = META_DIR / f'meta_{sess_name}.joblib'
    if not p.exists():
        ax.set_title(f'{sess_name} — not trained', color='white')
        continue

    bundle     = joblib.load(p)
    model      = bundle['model']
    feat_names = bundle['meta_feature_cols']
    importance = pd.Series(model.feature_importances_, index=feat_names)
    top20      = importance.sort_values(ascending=True).tail(20)
    colors     = ['#ff6b81' if f in quantile_feats else '#4fc3f7' for f in top20.index]

    ax.barh(top20.index, top20.values, color=colors, alpha=0.8)
    n_rows = bundle['n_train_rows']
    cv_auc = bundle['cv_auc']
    ax.set_title(f'{sess_name.upper()} meta\nCV AUC={cv_auc:.4f} | {n_rows:,} rows\n(red=quantile-derived)',
                 color='white', fontsize=10)

plt.suptitle('Feature Importance — Session-Specific Meta Models', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print('=' * 70)
print('SESSION-SPECIFIC META MODELS — SUMMARY')
print('=' * 70)

print(f'\nModels saved to: {META_DIR.resolve()}')
for f in sorted(META_DIR.glob('*.joblib')):
    b = joblib.load(f)
    print(f'  {f.name:<30} CV AUC={b["cv_auc"]:.4f} | rows={b["n_train_rows"]:,} | iters={b["n_iters"]}')

print(f'\nBase quantile models: UNCHANGED from {QUANT_DIR}')

print(f'\n--- Best threshold on test set ---')
print(f'  Threshold: P > {BEST_THRESH:.2f}')
print(f'  Trades:    {best["trades"]:,}')
print(f'  Win Rate:  {best["wr"]:.1%}')
print(f'  EV/trade:  {best["ev"]:.6f}')
print(f'  Sharpe:    {best["sharpe"]:.2f}')

print(f'\nNext step: update capital simulation to route signals')
print(f'  through session-specific meta models from models_7/meta/')